In [1]:
import tubesml as tml
import pandas as pd
import numpy as np

from source.report import _point_to_proba

from sklearn.metrics import brier_score_loss, mean_squared_error

from sklearn.model_selection import KFold

from sklearn.linear_model import Ridge, LogisticRegression, Lasso
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

import optuna
from optuna.samplers import TPESampler

In [2]:
df = pd.read_csv('data/processed/full_training.csv')

N_FOLDS = 5
kfolds = KFold(n_splits=N_FOLDS, shuffle=True, random_state=13)

df_train, df_test = tml.make_test(df, test_size=0.2, random_state=34)

DROP = ["target", "target_points", "ID", "DayNum", "Team1", "Team2",
        'T1_Loc', 'T2_Loc',
                "T1_region", "T2_region", "Season", "delta_Loc",
                "Season", "competitive", "competitive_score",
                "delta_def_rating_diff", "delta_impact_diff",
                "T1_def_rating_diff", "T2_def_rating_diff"]

df_train.head()

df_train = df.copy()

## Feats cats

In [3]:
all_feats = [c for c in df_train if c not in DROP]
all_feats

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [4]:
all_delta = [c for c in df_train if c not in DROP and "delta" in c]

all_delta

['delta_Ast',
 'delta_Ast_TO_ratio',
 'delta_Ast_TO_ratio_diff',
 'delta_Ast_diff',
 'delta_Away',
 'delta_Blk',
 'delta_Blk_diff',
 'delta_DR',
 'delta_DR_diff',
 'delta_DR_opportunity',
 'delta_DR_opportunity_diff',
 'delta_Eff_FG_perc_diff',
 'delta_FG3_ratio',
 'delta_FG3_ratio_diff',
 'delta_FGA',
 'delta_FGA2',
 'delta_FGA2_diff',
 'delta_FGA3',
 'delta_FGA3_diff',
 'delta_FGA_diff',
 'delta_FGM',
 'delta_FGM2',
 'delta_FGM2_diff',
 'delta_FGM3',
 'delta_FGM3_diff',
 'delta_FGM_diff',
 'delta_FGM_no_ast',
 'delta_FGM_no_ast_diff',
 'delta_FTA',
 'delta_FTA_diff',
 'delta_FTM',
 'delta_FTM_diff',
 'delta_N_wins',
 'delta_OR',
 'delta_OR_diff',
 'delta_OR_opportunity',
 'delta_OR_opportunity_diff',
 'delta_OT_win',
 'delta_PF',
 'delta_PF_diff',
 'delta_Score',
 'delta_Score_diff',
 'delta_Stl',
 'delta_Stl_diff',
 'delta_TO',
 'delta_TO_diff',
 'delta_TO_perposs',
 'delta_TO_perposs_diff',
 'delta_Tot_Reb',
 'delta_Tot_Reb_diff',
 'delta_True_shooting_perc_diff',
 'delta_def_ratin

In [5]:
no_delta = [c for c in df_train if c not in DROP and "delta" not in c]
no_delta

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [6]:
seeds = [c for c in df_train if c not in DROP and "Seed" in c] + [c for c in df_train if "quality" in c] + [c for c in df_train if "stage" in c] + [c for c in df_train if "elo" in c] + ["tourney"]
seeds

['T1_Seed',
 'T2_Seed',
 'delta_Seed',
 'T1_quality',
 'T2_quality',
 'delta_quality',
 'stage_Round1',
 'stage_Round2',
 'stage_Round3',
 'stage_Round4',
 'stage_final',
 'stage_finalfour',
 'stage_impossible',
 'T1_elo',
 'T2_elo',
 'delta_elo',
 'tourney']

In [7]:
no_seeds = [c for c in df_train if c not in DROP and "Seed" not in c]
no_seeds

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [8]:
feats_dict = {"all_feats": all_feats,
              "all_delta": all_delta, "no_delta": no_delta, "no_seeds": no_seeds, "seeds": seeds}

# Points predictions

## LGBM

In [9]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMRegressor(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                              learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="l2")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "l2"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [10]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

[I 2026-03-08 08:39:33,436] A new study created in memory with name: no-name-3521ce27-d184-45fd-afde-4480879ab716
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
in

Number of finished trials: 1000
Best trial: {'max_depth': 90, 'num_leaves': 13, 'reg_lambda': 11.865936878578008, 'reg_alpha': 20.532514349965357, 'colsample_bytree': 0.46526214114851555, 'subsample': 0.4405185534411223, 'min_child_weight': 4.662880615219311, 'feats': 'all_delta', 'clip_val': 23, 'padd': 0.04124760745742165}


In [11]:
0.183152

0.183152

In [12]:
study.trials_dataframe().sort_values('value', ascending=True).head(20)

,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_num_leaves,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
896,896,0.165223,2026-03-08 09:08:38.045697,2026-03-08 09:09:02.478910,0 days 00:00:24.433213,23,0.465262,all_delta,90,4.662881,13,0.041248,20.532514,11.865937,0.440519,COMPLETE
919,919,0.165267,2026-03-08 09:09:10.207504,2026-03-08 09:09:32.854743,0 days 00:00:22.647239,22,0.443757,all_delta,74,0.665615,10,0.038369,22.466277,8.727274,0.677269,COMPLETE
657,657,0.165284,2026-03-08 09:02:25.944359,2026-03-08 09:02:44.463992,0 days 00:00:18.519633,25,0.461739,all_delta,51,13.993919,13,0.042001,33.951447,15.813333,0.486228,COMPLETE
757,757,0.165286,2026-03-08 09:05:07.247985,2026-03-08 09:05:28.257015,0 days 00:00:21.009030,23,0.518415,all_delta,72,56.215584,13,0.035877,7.276913,5.322231,0.942060,COMPLETE
409,409,0.165321,2026-03-08 08:56:12.807364,2026-03-08 08:56:40.065992,0 days 00:00:27.258628,24,0.473596,all_delta,62,18.760913,12,0.044137,30.764173,13.639863,0.862510,COMPLETE
986,986,0.165337,2026-03-08 09:10:49.061534,2026-03-08 09:11:12.212394,0 days 00:00:23.150860,20,0.411780,all_delta,90,0.264160,16,0.032475,21.489488,2.411309,0.673558,COMPLETE
965,965,0.165358,2026-03-08 09:10:19.624973,2026-03-08 09:10:42.162861,0 days 00:00:22.537888,20,0.461415,all_delta,94,1.114113,12,0.035620,17.941856,9.856678,0.630014,COMPLETE
961,961,0.165361,2026-03-08 09:10:13.771342,2026-03-08 09:10:34.160245,0 days 00:00:20.388903,21,0.460947,all_delta,89,0.303680,12,0.035044,20.699644,10.124168,0.694314,COMPLETE
797,797,0.165367,2026-03-08 09:06:10.228562,2026-03-08 09:06:31.632664,0 days 00:00:21.404102,24,0.533478,all_delta,81,37.805883,15,0.035130,11.110234,4.328134,0.416894,COMPLETE
706,706,0.165383,2026-03-08 09:03:45.219791,2026-03-08 09:04:03.672995,0 days 00:00:18.453204,24,0.683460,all_delta,42,28.275477,12,0.037152,27.811190,13.944548,0.403106,COMPLETE


## XGBoost

In [13]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBRegressor(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            early_stopping_rounds=100,
                             eval_metric=mean_squared_error)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {'verbose': False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [14]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 1000
Best trial: {'max_depth': 3, 'reg_lambda': 12.738494759481998, 'reg_alpha': 19.202063857291815, 'colsample_bytree': 0.970208060904755, 'colsample_bylevel': 0.6628404550150686, 'subsample': 0.9348070408543836, 'min_child_weight': 124.29945187980685, 'feats': 'all_delta', 'clip_val': 20, 'padd': 0.00905924560132836}


,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bylevel,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
777,777,0.165296,2026-03-08 12:09:45.724814,2026-03-08 12:11:23.548965,0 days 00:01:37.824151,20,0.662840,0.970208,all_delta,3,124.299452,0.009059,19.202064,12.738495,0.934807,COMPLETE
764,764,0.165388,2026-03-08 12:08:02.307055,2026-03-08 12:09:45.718432,0 days 00:01:43.411377,22,0.667144,0.968823,all_delta,3,132.220532,0.008391,19.009442,18.966694,0.975144,COMPLETE
775,775,0.165637,2026-03-08 12:09:41.688801,2026-03-08 12:11:18.261612,0 days 00:01:36.572811,20,0.665183,0.947559,all_delta,3,122.234421,0.008894,18.480501,18.879752,0.938444,COMPLETE
854,854,0.165718,2026-03-08 12:25:40.292103,2026-03-08 12:27:01.887822,0 days 00:01:21.595719,20,0.681514,0.965278,all_delta,3,156.253200,0.008384,20.565067,19.520474,0.937520,COMPLETE
953,953,0.165731,2026-03-08 12:40:29.692949,2026-03-08 12:41:50.616555,0 days 00:01:20.923606,22,0.675067,0.948704,all_delta,3,161.000716,0.008634,18.168024,18.824047,0.905328,COMPLETE
723,723,0.165749,2026-03-08 12:00:19.413747,2026-03-08 12:01:27.623201,0 days 00:01:08.209454,23,0.696401,0.939500,all_delta,3,134.425982,0.008521,19.951462,17.570152,0.963089,COMPLETE
598,598,0.165763,2026-03-08 11:38:41.636095,2026-03-08 11:39:56.226013,0 days 00:01:14.589918,21,0.674696,0.975790,all_delta,3,112.467238,0.041078,27.854238,17.877329,0.941530,COMPLETE
766,766,0.165831,2026-03-08 12:08:16.431628,2026-03-08 12:09:54.613046,0 days 00:01:38.181418,22,0.666233,0.968376,all_delta,3,144.461179,0.009659,18.216983,19.014008,0.974077,COMPLETE
659,659,0.165837,2026-03-08 11:49:02.796560,2026-03-08 11:50:57.333267,0 days 00:01:54.536707,22,0.640222,0.950950,all_delta,3,115.252742,0.009294,30.221988,22.245074,0.946182,COMPLETE
852,852,0.165841,2026-03-08 12:25:32.368319,2026-03-08 12:26:54.178006,0 days 00:01:21.809687,20,0.684968,0.967236,all_delta,3,159.855276,0.008167,22.130900,20.310193,0.933842,COMPLETE


## Ridge

In [15]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Ridge(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [16]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 2000
Best trial: {'alpha': 10.648932219848346, 'feats': 'all_delta', 'clip_val': 23, 'padd': 0.01971647622688984}


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
1530,1530,0.165357,2026-03-08 13:20:46.765302,2026-03-08 13:21:02.481146,0 days 00:00:15.715844,10.648932,23,all_delta,0.019716,COMPLETE
1595,1595,0.165358,2026-03-08 13:21:50.244070,2026-03-08 13:22:09.017067,0 days 00:00:18.772997,10.813360,23,all_delta,0.010444,COMPLETE
581,581,0.165358,2026-03-08 13:04:13.021250,2026-03-08 13:04:24.769390,0 days 00:00:11.748140,10.798193,23,all_delta,0.008961,COMPLETE
1585,1585,0.165358,2026-03-08 13:21:35.300474,2026-03-08 13:21:49.795862,0 days 00:00:14.495388,10.705163,23,all_delta,0.010687,COMPLETE
610,610,0.165358,2026-03-08 13:05:04.437008,2026-03-08 13:05:22.648856,0 days 00:00:18.211848,10.673745,23,all_delta,0.011586,COMPLETE
613,613,0.165358,2026-03-08 13:05:06.614007,2026-03-08 13:05:23.822798,0 days 00:00:17.208791,10.662603,23,all_delta,0.011468,COMPLETE
909,909,0.165358,2026-03-08 13:10:16.346384,2026-03-08 13:10:32.479001,0 days 00:00:16.132617,10.878121,23,all_delta,0.011148,COMPLETE
588,588,0.165358,2026-03-08 13:04:22.896701,2026-03-08 13:04:37.027466,0 days 00:00:14.130765,10.609078,23,all_delta,0.008677,COMPLETE
1779,1779,0.165358,2026-03-08 13:25:15.503826,2026-03-08 13:25:28.680716,0 days 00:00:13.176890,10.517503,23,all_delta,0.012581,COMPLETE
974,974,0.165358,2026-03-08 13:11:14.881629,2026-03-08 13:11:31.202478,0 days 00:00:16.320849,10.899163,23,all_delta,0.009032,COMPLETE


## Lasso

In [17]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Lasso(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [18]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
invalid value encountered in cast


Number of finished trials: 2000
Best trial: {'alpha': 0.11287575902911327, 'feats': 'all_delta', 'clip_val': 23, 'padd': 0.011351441200606549}


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
1161,1161,0.165694,2026-03-08 13:55:23.183678,2026-03-08 13:55:35.999400,0 days 00:00:12.815722,0.112876,23,all_delta,0.011351,COMPLETE
845,845,0.165694,2026-03-08 13:49:52.451577,2026-03-08 13:50:09.453135,0 days 00:00:17.001558,0.111643,23,all_delta,0.009665,COMPLETE
1558,1558,0.165694,2026-03-08 14:02:15.402462,2026-03-08 14:02:28.330290,0 days 00:00:12.927828,0.114506,23,all_delta,0.014486,COMPLETE
1850,1850,0.165694,2026-03-08 14:07:38.288694,2026-03-08 14:07:52.956723,0 days 00:00:14.668029,0.114406,23,all_delta,0.018241,COMPLETE
840,840,0.165695,2026-03-08 13:49:48.946002,2026-03-08 13:50:02.472382,0 days 00:00:13.526380,0.114854,23,all_delta,0.009461,COMPLETE
1914,1914,0.165696,2026-03-08 14:08:44.262955,2026-03-08 14:09:01.539452,0 days 00:00:17.276497,0.116082,23,all_delta,0.017233,COMPLETE
1565,1565,0.165696,2026-03-08 14:02:21.423139,2026-03-08 14:02:37.740094,0 days 00:00:16.316955,0.109061,23,all_delta,0.015129,COMPLETE
1590,1590,0.165697,2026-03-08 14:02:42.332527,2026-03-08 14:02:57.208235,0 days 00:00:14.875708,0.110366,24,all_delta,0.014855,COMPLETE
1345,1345,0.165699,2026-03-08 13:58:22.122283,2026-03-08 13:58:34.132191,0 days 00:00:12.009908,0.113487,22,all_delta,0.009983,COMPLETE
701,701,0.165700,2026-03-08 13:47:12.055586,2026-03-08 13:47:28.223876,0 days 00:00:16.168290,0.112434,22,all_delta,0.013996,COMPLETE


# Probability Predictions


## LGBM

In [19]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMClassifier(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                               learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "auc"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [20]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 1000
Best trial: {'max_depth': 195, 'num_leaves': 21, 'reg_lambda': 21.390761919810387, 'reg_alpha': 0.06309806111703037, 'colsample_bytree': 0.882386981850899, 'subsample': 0.9661210930098577, 'min_child_weight': 1.7775885322232414, 'feats': 'all_feats'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,state
436,436,0.165835,2026-03-08 14:22:13.700139,2026-03-08 14:22:58.683136,0 days 00:00:44.982997,0.882387,all_feats,195,1.777589,21,0.063098,21.390762,0.966121,COMPLETE
355,355,0.165855,2026-03-08 14:18:41.197084,2026-03-08 14:19:21.432051,0 days 00:00:40.234967,0.793341,all_feats,193,1.189624,15,2.078816,19.091115,0.777693,COMPLETE
386,386,0.165909,2026-03-08 14:20:02.578265,2026-03-08 14:20:50.800109,0 days 00:00:48.221844,0.965658,all_feats,186,0.019295,23,0.677089,22.012978,0.958314,COMPLETE
962,962,0.166003,2026-03-08 14:43:50.955764,2026-03-08 14:44:30.401547,0 days 00:00:39.445783,0.931712,all_feats,3,9.766706,18,0.018171,18.649508,0.513602,COMPLETE
354,354,0.166010,2026-03-08 14:18:39.718850,2026-03-08 14:19:08.637630,0 days 00:00:28.918780,0.837323,all_feats,3,15.282607,18,2.145178,17.342192,0.993225,COMPLETE
513,513,0.166025,2026-03-08 14:25:45.322543,2026-03-08 14:26:21.758388,0 days 00:00:36.435845,0.890595,all_feats,196,0.448567,20,6.754181,16.638792,0.890751,COMPLETE
393,393,0.166119,2026-03-08 14:20:31.492299,2026-03-08 14:21:14.759657,0 days 00:00:43.267358,0.893751,all_feats,190,4.095319,22,0.128613,22.393022,0.957760,COMPLETE
525,525,0.166122,2026-03-08 14:26:14.131551,2026-03-08 14:27:04.973791,0 days 00:00:50.842240,0.959297,all_feats,184,0.173434,22,0.253634,26.321937,0.975615,COMPLETE
426,426,0.166147,2026-03-08 14:21:58.709603,2026-03-08 14:22:42.564953,0 days 00:00:43.855350,0.884036,all_feats,195,0.517756,22,0.200707,21.801640,0.928085,COMPLETE
304,304,0.166154,2026-03-08 14:17:08.983415,2026-03-08 14:17:35.068545,0 days 00:00:26.085130,0.736141,all_delta,182,18.782104,15,5.372485,13.957803,0.862351,COMPLETE


## XGBoost

In [21]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBClassifier(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            early_stopping_rounds=100,
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {"verbose": False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

## LogisticRegression

In [ ]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        'C': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = LogisticRegression(C=param["C"], random_state=34, max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=500, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)